In [ ]:
# ── Clone repo from GitHub (run once per Colab session) ───────────────────
import os, sys
if not os.path.exists('/content/protosearch'):
    !git clone https://github.com/bjdraper/protosearch.git /content/protosearch
sys.path.insert(0, '/content/protosearch')
os.chdir('/content/protosearch')
!pip install -q -r /content/protosearch/requirements.txt

# 00 — Setup
**Run this once before any other notebook.**

This notebook:
1. Installs all required tools and Python packages
2. Mounts Google Drive (or sets a local working directory)
3. Asks you to define your protein family (reference sequences + domain HMM)
4. Downloads your reference probes from UniProt and the HMM profile from Pfam

---
### What you need to provide
| Input | Where | Required? |
|---|---|---|
| Reference sequences | UniProt accessions **or** a FASTA file | ✅ |
| Input proteomes | A `.faa` file or NCBI taxon IDs | ✅ |
| Domain HMM | Pfam accession (auto-downloaded) **or** custom `.hmm` file | ✅ |
| Key residue motifs | Regex patterns (e.g. `H.{2}D`) | Optional |
| AlphaFold targets | UniProt accessions | Optional |

In [ ]:
# ── STEP 1 & 2: Install system tools and Python packages ─────────────────────
!apt-get install -qq hmmer mafft fasttree
!pip install -q numpy pandas scipy scikit-learn biopython \
    fair-esm faiss-cpu leidenalg python-igraph openTSNE \
    ete3 logomaker py3Dmol transformers accelerate seaborn pyyaml tqdm

In [ ]:
# (merged into Step 1 above)

In [ ]:
# ── STEP 3: Set working directory ─────────────────────────────────────────────
import os, sys

USE_DRIVE = True   # False if running locally

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/agpat_crustacea'
else:
    PROJECT_ROOT = '/content/protosearch'

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, '/content/protosearch')
print('Working directory:', os.getcwd())

In [ ]:
# ── STEP 4: Define your protein family ───────────────────────────────────────
import shutil

SURVEY_NAME = 'agpat_crustacea'

REFERENCE_PROBES = [
    ('AGPAT1_HUMAN',   'Q99943', 'AGPAT1',             '#D94040'),
    ('AGPAT2_HUMAN',   'O15120', 'AGPAT2',             '#E05C5C'),
    ('AGPAT3_HUMAN',   'Q9NRZ7', 'AGPAT3',             '#E87878'),
    ('AGPAT4_HUMAN',   'Q9NRZ5', 'AGPAT4',             '#F09090'),
    ('AGPAT5_HUMAN',   'Q9NUQ2', 'AGPAT5',             '#F8B0B0'),
    ('LCLAT1_HUMAN',   'Q6UWP7', 'LCLAT1',             '#F57F17'),
    ('LCLAT1_MOUSE',   'Q6NVG1', 'LCLAT1 (mouse)',     '#E65100'),
    ('LCLAT1_DANRE',   'Q6NYV8', 'LCLAT1 (zebrafish)', '#FB8C00'),
    ('GPAT4_HUMAN',    'Q86UL3', 'GPAT4',              '#00695C'),
    ('GPAT3_HUMAN',    'Q53EU6', 'GPAT3',              '#2E7D32'),
    ('GPAT1_MOUSE',    'Q61586', 'GPAT1 (mouse)',      '#004D40'),
    ('GPAT2_MOUSE',    'Q8BYM8', 'GPAT2 (mouse)',      '#1B5E20'),
    ('AGPAT2_MOUSE',   'Q8K3K7', 'AGPAT2 (mouse)',     '#B71C1C'),
    ('LPCAT2_HUMAN',   'Q7L5N7', 'LPCAT2',             '#1565C0'),
    ('PlsC_ECOLI',     'P26647', 'PlsC (E.coli)',      '#388E3C'),
    ('PlsC_BACSU',     'O34478', 'PlsC (B.sub)',       '#33691E'),
    ('PlsC_THAPS',     'B8BZR1', 'PlsC (diatom)',      '#558B2F'),
    ('Tafazzin_HUMAN', 'Q16635', 'Tafazzin [ctrl]',   '#4A148C'),
    ('LPCAT3_HUMAN',   'Q6P1A2', 'LPCAT3 [ctrl]',     '#B0BEC5'),
    ('MBOAT7_HUMAN',   'Q96N66', 'MBOAT7 [ctrl]',     '#90A4AE'),
]

PFAM_IDS = ['PF01553']   # acyltransferase domain

MIN_LENGTH = 200
MAX_LENGTH = 500

ALPHAFOLD_TARGETS = {
    'AGPAT1_HUMAN':   'Q99943',
    'AGPAT2_HUMAN':   'O15120',
    'AGPAT3_HUMAN':   'Q9NRZ7',
    'AGPAT4_HUMAN':   'Q9NRZ5',
    'AGPAT5_HUMAN':   'Q9NUQ2',
    'LCLAT1_HUMAN':   'Q6UWP7',
    'GPAT4_HUMAN':    'Q86UL3',
    'GPAT3_HUMAN':    'Q53EU6',
    'LPCAT2_HUMAN':   'Q7L5N7',
    'Tafazzin_HUMAN': 'Q16635',
}

CATALYTIC_MOTIFS = {
    'HxxD': {'pattern': 'H.{2}D', 'colour': '#FF6B00', 'role': 'catalytic dyad'},
    'FPxG': {'pattern': 'FP.G',   'colour': '#FFD700', 'role': 'acyl-CoA binding'},
    'EGTR': {'pattern': 'EGTR',   'colour': '#00CED1', 'role': 'Block II conserved'},
}

# Copy pre-built config from repo
shutil.copy('/content/protosearch/config.yaml', 'config.yaml')

print(f'Survey: {SURVEY_NAME}')
print(f'Reference probes: {len(REFERENCE_PROBES)}')
print(f'Pfam IDs: {PFAM_IDS}')

In [ ]:
# ── STEP 5: Write config.yaml from your inputs ────────────────────────────────
import yaml

config = {
    'paths': {
        'data_dir':    'data',
        'results_dir': 'results',
    },
    'reference_probes': [
        {'id': pid, 'acc': acc, 'label': label, 'colour': colour}
        for pid, acc, label, colour in REFERENCE_PROBES
    ],
    'alphafold_targets': ALPHAFOLD_TARGETS,
    'filter':   {'min_length': MIN_LENGTH, 'max_length': MAX_LENGTH},
    'hmmer':    {'evalue': 1e-5, 'cpu': 4, 'profiles': PFAM_IDS},
    'embedding': {'model': 'esm2_t33_650M_UR50D', 'layer': 33, 'dim': 1280,
                  'batch_size': 32, 'device': 'cuda'},
    'clustering':    {'k_neighbors': 25, 'resolution': 2.0, 'pca_dims': 50, 'tsne_perp': 100},
    'subclustering': {},
    'tree':      {'mafft_flags': '--auto --thread 8 --quiet', 'fasttree_model': 'lg',
                  'iqtree_model': 'LG+G4', 'iqtree_bootstrap': 1000, 'iqtree_threads': 8},
    'structure': {'esmfold_model': 'facebook/esmfold_v1'},
    'catalytic_motifs': CATALYTIC_MOTIFS,
}

with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print('config.yaml written.')
print('You can edit it directly at any time to tune clustering or tree parameters.')

In [ ]:
# ── STEP 6: Download reference probes from UniProt ────────────────────────────
from acyltransferase.config import load_config
from acyltransferase.utils  import fetch_uniprot_sequence, write_fasta

cfg = load_config('config.yaml')

probe_dir = 'data/query_sequences'
os.makedirs(probe_dir, exist_ok=True)
probes_faa = f'{probe_dir}/reference_probes.faa'

if not os.path.exists(probes_faa):
    records = []
    for probe in cfg.reference_probes:
        try:
            _, seq = fetch_uniprot_sequence(probe['acc'])
            records.append((probe['id'], seq))
            print(f"  {probe['id']}: {len(seq)} aa")
        except Exception as e:
            print(f"  {probe['id']}: FAILED — {e}")
    write_fasta(records, probes_faa)
    print(f'Saved {len(records)} probes → {probes_faa}')
else:
    from acyltransferase.utils import read_fasta
    print(f'Probes already present: {len(read_fasta(probes_faa))} sequences')

In [ ]:
# ── STEP 7: Download Pfam HMM profile ────────────────────────────────────────
from acyltransferase.search import download_hmm_profile

hmm_dir = 'data/hmm_profiles'
os.makedirs(hmm_dir, exist_ok=True)

for pfam_id in cfg.hmmer['profiles']:
    out = f'{hmm_dir}/{pfam_id}.hmm'
    if not os.path.exists(out):
        print(f'Downloading {pfam_id} ...')
        download_hmm_profile(pfam_id, out)
    else:
        print(f'{pfam_id}.hmm already present')

!hmmpress {hmm_dir}/*.hmm 2>/dev/null || true
print('Done.')

## ✓ Setup complete

**Next steps:**
1. **Upload your proteome** — put your `.faa` file at `data/proteins_raw/proteins.faa`  
   *(or provide NCBI taxon IDs and use the fetch utilities in `acyltransferase.utils`)*
2. Open **`01_embed_search.ipynb`** and set `YOUR_FASTA` to your file path
3. Run notebooks in order: `01` → `02` → `03` → `04`

---
**To change parameters later**, edit `config.yaml` directly — no need to re-run this notebook.